In [3]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

In [4]:
os.makedirs("dataset_outputs/A2AR/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="A2AR/data/a2ar_train_1",
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [5]:
display(dataset.X.shape)
display(dataset.X_ind.shape)
display(dataset.getDF())
display(dataset.X)


(2407, 0)

(0, 0)

,QSPRID,Y,Drug,Y_original
QSPRID,,,,
A2ARDataset_0000,A2ARDataset_0000,True,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,True
A2ARDataset_0001,A2ARDataset_0001,True,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,True
A2ARDataset_0002,A2ARDataset_0002,False,COc1nc(N)c(C#N)c(-c2ccc3c(c2)OCO3)c1C#N,False
A2ARDataset_0003,A2ARDataset_0003,True,CCNC(=O)C1OC(n2cnc3c(NCC)nc(C#CCCCc4ccccc4)nc3...,True
A2ARDataset_0004,A2ARDataset_0004,True,Cc1cc(C)n(-c2cc(NC(=O)CN3CCOCC3)nc(-c3ccc(C)o3...,True
...,...,...,...,...
A2ARDataset_2402,A2ARDataset_2402,True,CCCn1cc2c(nc(NC(=O)Nc3ccccc3OC)n3nc(-c4ccco4)n...,True
A2ARDataset_2403,A2ARDataset_2403,True,O=C(COc1ccc(-c2cc3c([nH]2)c(=O)n(CC2CC2)c(=O)n...,True
A2ARDataset_2404,A2ARDataset_2404,True,CNc1ncc(C(=O)NCc2ccc(OC)cc2)c2nc(-c3ccco3)nn12,True


""
QSPRID
A2ARDataset_0000
A2ARDataset_0001
A2ARDataset_0002
A2ARDataset_0003
A2ARDataset_0004
...
A2ARDataset_2402
A2ARDataset_2403
A2ARDataset_2404


In [6]:
import torch
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset

# Nastavení zařízení (GPU nebo CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načtení tokenizeru a modelu
tokenizer = RobertaTokenizerFast.from_pretrained("entropy/roberta_zinc_480m", max_len=128)
model = RobertaForMaskedLM.from_pretrained('entropy/roberta_zinc_480m')

# Přenesení modelu na správné zařízení
model.to(device)

# Připravení collatoru pro padding
collator = DataCollatorWithPadding(tokenizer, padding=True, return_tensors='pt')

# Načtení seznamu SMILES (například z nějaké jiné struktury než datasetu)
smiles = dataset.getDF()["Drug"]

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding

smiles_dataset = SMILESDataset(smiles, tokenizer)

# Vytvoření DataLoaderu (dávky po 32)
batch_size = 32
dataloader = DataLoader(smiles_dataset, batch_size=batch_size, collate_fn=collator)

# Zpracování dat po dávkách
model.eval()  # Převede model do evaluačního režimu (bez trénování)
embeddings_list = []  # Uchováme všechny embeddings

with torch.no_grad():  # Nevytvářet gradienty během evaluace
    for batch in dataloader:
        # Zkontroluj tvar batchů
        print("Batch input_ids tvar:", batch['input_ids'].shape)
        print("Batch attention_mask tvar:", batch['attention_mask'].shape)

        # Přenesení všech vstupů na správné zařízení (GPU nebo CPU)
        input_ids = batch['input_ids'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi
        attention_mask = batch['attention_mask'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi

        # Modelování výstupů s hidden states
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # Získání poslední vrstvy hidden states
        full_embeddings = outputs[1][-1]

        # Výpočet průměrného embeddingu pro každý token
        embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
        
        # Uložení embeddings pro tuto dávku
        embeddings_list.append(embeddings)

# Spojení všech embeddings z dávky do jednoho tensoru
all_embeddings = torch.cat(embeddings_list, dim=0)

# Teď můžeš použít `all_embeddings`, což obsahuje embeddings pro všechny SMILES v datasetu


/tmp/ipykernel_21007/2991385132.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch inpu

In [7]:
import pandas as pd

# Převod na NumPy pole a pak na DataFrame
df = pd.DataFrame(all_embeddings.cpu().numpy())


In [8]:
dataset.X = pd.concat([dataset.X, df], axis=1)


/tmp/ipykernel_21007/1146750015.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  dataset.X = pd.concat([dataset.X, df], axis=1)


In [9]:
display(dataset.X.shape)

(4814, 768)

In [10]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [11]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("A2AR/data/a2ar_train_1")

X2_all = load_datasets("A2AR/data/a2ar_val_1")

X3_all = load_datasets("A2AR/data/a2ar_test_1")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_21007/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [13]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [14]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [15]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [16]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [17]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [18]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.336137,0.421905,-1.816749,-0.037482,1.781845,-1.550991,-1.025524,0.352857,0.128960,-1.762285
1,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.908652,-0.195387,1.484141,0.551087,-0.041965,0.914704,1.212196,0.120372,0.309187,-1.621375
2,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.625598,-0.418360,0.122820,-2.978834,-1.065097,2.712647,0.613691,-1.042389,-2.925378,0.038545
3,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-1.365275,0.231594,0.168931,1.069064,-0.871705,-0.972357,-0.970038,1.735897,-0.877857,-0.114969
4,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.122059,0.693463,-1.244037,-0.024998,0.976028,-1.697487,-1.362892,0.355531,-0.253875,-1.866983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3470,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,1.499616,-1.711053,-0.720884,2.738597,-0.069114,-0.121176,2.573974,-0.270240,-0.972412,1.562329
3471,-0.096043,0.333173,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.997047,1.388580,0.662801,-0.510915,-0.073312,-0.765406,-0.377708,1.110718,-2.273164,0.152155
3472,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.233030,0.049356,-1.458684,0.166379,-0.644038,1.236481,0.245435,1.333642,0.729677,-1.898828
3473,-0.096043,-0.309168,-0.093815,-0.108488,-0.07369,-0.04999,-0.070784,-0.731489,-0.144154,-0.24376,...,-0.871648,0.274557,-0.184794,1.130070,1.533618,-1.370479,-1.527590,1.690123,-0.229953,-0.931566


In [19]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [20]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [21]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim
def objective(trial, X_train, y_train, X_test, y_test):
    dropout_frac = trial.suggest_categorical("dropout_frac", [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    batch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", ["optim.AdamW", "optim.RMSprop"])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    neuron_layers_dict = {
    '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 1024, 256, 64, 8]': [4096, 1024, 256, 64, 8],
    '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]': [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
    '[200]': [200],
    '[2000]': [2000],
    '[2000, 1000]': [2000, 1000],
    '[2000, 1000, 500]': [2000, 1000, 500],
    '[1000, 50]': [1000, 50],
    '[4000, 2000]': [4000, 2000],
    '[4000, 2000, 1000, 500]': [4000, 2000, 1000, 500],
    '[4000, 2000, 2000, 500]': [4000, 2000, 2000, 500]
    }
    neuron_layers_size = trial.suggest_categorical("neuron_layers_size", list(neuron_layers_dict.keys()))
    opt = {"optim.AdamW": optim.AdamW,
           "optim.RMSprop": optim.RMSprop}
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=tol,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers= neuron_layers_dict[neuron_layers_size],  # Použití neuron_layers_size
        batch_size=batch_size,
        optimizer=opt[optimizer],
        lr=lr,
        random_seed=69
    )

    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)

    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)

    return mcc  # maximalizujeme MCC


In [ ]:
study_3 = optuna.create_study(
    study_name="A2AR_study_bert",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=30
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-04-25 10:29:00,980] A new study created in RDB with name: A2AR_study_bert


cuda


[I 2025-04-25 10:35:00,019] Trial 0 finished with value: 0.1023886610505542 and parameters: {'dropout_frac': 0.4, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.1, 'n_epochs': 500, 'batch_size': 64, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]'}. Best is trial 0 with value: 0.1023886610505542.


cuda


[I 2025-04-25 10:38:15,424] Trial 1 finished with value: 0.0 and parameters: {'dropout_frac': 0.1, 'patience': 75, 'tol': 0.01, 'weight_decay': 0.0001, 'n_epochs': 1000, 'batch_size': 512, 'optimizer': 'optim.RMSprop', 'lr': 0.01, 'neuron_layers_size': '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]'}. Best is trial 0 with value: 0.1023886610505542.


...

cuda


[I 2025-04-25 11:24:00,350] Trial 21 finished with value: 0.20961729257309258 and parameters: {'dropout_frac': 0.6, 'patience': 10, 'tol': 0.01, 'weight_decay': 1e-06, 'n_epochs': 300, 'batch_size': 256, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[200]'}. Best is trial 16 with value: 0.29659766368726764.


cuda


[I 2025-04-25 11:33:32,207] Trial 22 finished with value: 0.17849243801855255 and parameters: {'dropout_frac': 0.2, 'patience': 10, 'tol': 0.001, 'weight_decay': 1e-05, 'n_epochs': 1000, 'batch_size': 64, 'optimizer': 'optim.AdamW', 'lr': 1e-05, 'neuron_layers_size': '[4000, 2000, 2000, 500]'}. Best is trial 16 with value: 0.29659766368726764.


cuda


In [20]:
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))


CUDA_LAUNCH_BLOCKING = 1
